# Fintech P1 — Week 1 Homework: The Overfitting Trap in Marketing ROI

This is a **scaffold, not a solution**. The plumbing is done (imports, data load,
train/test split, a metrics helper). Each homework step is marked `TODO` with a
pointer to the lecture cell that shows the same pattern.

## How to use this notebook

- `Shift+Enter` runs a cell and moves to the next one; `Ctrl+Enter` runs it and stays.
- Work **top to bottom** the first time — later cells need variables from earlier ones.
- If something behaves inexplicably, click **Restart** in the toolbar and run from the
  top. Stale variables from an earlier run are the usual culprit.
- Written answers go in the `ANSWER` cells. Keep them in the notebook: **this file is
  your homework submission.**

Your reference for every pattern below is `lecture_week1.py`.


# Setup: imports

Everything the homework needs, imported once at the top. This is the normal
way to write a .py script (in the notebook the imports were scattered across
the parts, which is a notebook habit, not a Python one).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from math import sqrt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

# Setup: the scoring helper

Same idea as `score()` in the lecture notebook: results live in a dict keyed
by model name, so re-running a cell overwrites that model's row instead of
quietly appending a duplicate.

In [2]:
scores = {}


def score(name, train_pred, test_pred):
    """Score a model on the training sample and on the held-out test sample."""
    scores[name] = {
        'Model': name,
        'R-squared train': r2_score(y_train, train_pred),
        'R-squared test': r2_score(y_test, test_pred),
        'RMSE train': sqrt(mean_squared_error(y_train, train_pred)),
        'RMSE test': sqrt(mean_squared_error(y_test, test_pred)),
        'MAE train': mean_absolute_error(y_train, train_pred),
        'MAE test': mean_absolute_error(y_test, test_pred),
    }
    return pd.DataFrame(scores.values())


def results_table():
    """All models scored so far, side by side."""
    return pd.DataFrame(scores.values())

# Part 1: The "Simple" Model (Baseline)

## 1.1 Load the data

`index_col=0` matters: the file carries an unnamed first column that is just
the row number. Forget it and you silently train on it as a feature.

In [3]:
URL = ('https://raw.githubusercontent.com/JWarmenhoven/ISLR-python/'
       'master/Notebooks/Data/Advertising.csv')

advertising = pd.read_csv(URL, index_col=0)

print(advertising.head())
print(advertising.shape)
print(advertising.describe())

      TV  Radio  Newspaper  Sales
1  230.1   37.8       69.2   22.1
2   44.5   39.3       45.1   10.4
3   17.2   45.9       69.3    9.3
4  151.5   41.3       58.5   18.5
5  180.8   10.8       58.4   12.9
(200, 4)
               TV       Radio   Newspaper       Sales
count  200.000000  200.000000  200.000000  200.000000
mean   147.042500   23.264000   30.554000   14.022500
std     85.854236   14.846809   21.778621    5.217457
min      0.700000    0.000000    0.300000    1.600000
25%     74.375000    9.975000   12.750000   10.375000
50%    149.750000   22.900000   25.750000   12.900000
75%    218.825000   36.525000   45.100000   17.400000
max    296.400000   49.600000  114.000000   27.000000


## 1.2 Split into features and target, then train/test

The brief is explicit: `test_size=0.3`, `random_state=1`. The random_state
makes the split reproducible - without it you get a different split every run
and your numbers will not match what you wrote down.

In [4]:
X = advertising[['TV', 'Radio', 'Newspaper']]
y = advertising['Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

print(f"train: {X_train.shape[0]} rows")
print(f"test : {X_test.shape[0]} rows")

train: 140 rows
test : 60 rows


## 1.3 Fit the baseline LinearRegression

TODO: fit a plain `LinearRegression` on the three original features.
Pattern: lecture cell "Fitting the Linear Regression".
  - create the model, `.fit(X_train, y_train)`
  - predict on X_train and on X_test
  - `score('Baseline', <train prediction>, <test prediction>)`

In [5]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)

baseline_train_pred = baseline.predict(X_train)
baseline_test_pred = baseline.predict(X_test)

score('Baseline', baseline_train_pred, baseline_test_pred)

,Model,R-squared train,R-squared test,RMSE train,RMSE test,MAE train,MAE test
0,Baseline,0.885005,0.922461,1.789726,1.388857,1.374654,1.054833


## 1.4 Write down the coefficients

TODO: print the intercept and the coefficient on each of TV, Radio, Newspaper.
Hint: `pd.Series(baseline.coef_, index=X_train.columns)` gives you a labelled
series instead of a bare array - much easier to read and to quote in an answer.

In [6]:
print(f"Intercept: {baseline.intercept_:.4f}")

coefs = pd.Series(baseline.coef_, index=X_train.columns)
print(coefs)

Intercept: 2.9372
TV           0.046952
Radio        0.176586
Newspaper    0.001851
dtype: float64


ANSWER - Part 1 interpretation

Write, in the block below: what each coefficient means in business terms
(one extra unit of spend on that channel is associated with how much extra
Sales?), and the baseline train/test metrics.

In [7]:
"""
ANSWER (Part 1):

Coefficients (business interpretation):
- TV:        holding Radio and Newspaper fixed, one extra $1,000 of TV spend
             is associated with +0.0470 units (about 47 more units) of Sales.
- Radio:     holding TV and Newspaper fixed, one extra $1,000 of Radio spend
             is associated with +0.1766 units of Sales - roughly 4x more
             effective per dollar than TV.
- Newspaper: holding TV and Radio fixed, one extra $1,000 of Newspaper spend
             is associated with +0.0019 units of Sales - essentially flat,
             the smallest effect of the three by a wide margin.
- Intercept: with zero spend on all three channels, the model predicts
             2.9372 units of baseline Sales (e.g. from other channels/brand
             awareness not captured by these features).

Baseline train/test metrics (from results_table(), row 'Baseline'):
- R-squared train: 0.8850   R-squared test: 0.9225
- RMSE train:      1.7897   RMSE test:      1.3889
- MAE train:       1.3747   MAE test:       1.0548

Comment on train vs. test: test performance is actually *better* than train
here (higher R-squared, lower RMSE/MAE) - there is no overfitting gap at all,
if anything the test split happened to be a bit easier. This is the expected
behavior of a simple, low-variance model with only 3 features: it can't
memorize noise even if it wanted to. This is the number to keep in mind for
Part 2, where a much more flexible model will show the opposite pattern.
"""

"\nANSWER (Part 1):\n\nCoefficients (business interpretation):\n- TV:        holding Radio and Newspaper fixed, one extra $1,000 of TV spend\n             is associated with +0.0470 units (about 47 more units) of Sales.\n- Radio:     holding TV and Newspaper fixed, one extra $1,000 of Radio spend\n             is associated with +0.1766 units of Sales - roughly 4x more\n             effective per dollar than TV.\n- Newspaper: holding TV and Radio fixed, one extra $1,000 of Newspaper spend\n             is associated with +0.0019 units of Sales - essentially flat,\n             the smallest effect of the three by a wide margin.\n- Intercept: with zero spend on all three channels, the model predicts\n             2.9372 units of baseline Sales (e.g. from other channels/brand\n             awareness not captured by these features).\n\nBaseline train/test metrics (from results_table(), row 'Baseline'):\n- R-squared train: 0.8850   R-squared test: 0.9225\n- RMSE train:      1.7897   RMSE te

# Part 2: The "Overly Complex" Model (The Trap)

Do this part BY HAND (separate PolynomialFeatures -> StandardScaler ->
LinearRegression) so you see the order of operations. Parts 3 and 4 then use a
Pipeline, which does exactly this for you.

The rule that matters: `fit_transform` on the TRAINING data, `transform` only
on the test data. Fitting the transformer on the test set leaks information
from it into training and inflates every number you report afterwards.

## 2.1 Polynomial features, degree 5

TODO: build `PolynomialFeatures(degree=5, include_bias=False)`,
`fit_transform` on X_train, `transform` on X_test.
Pattern: lecture cell "Polynomial Transformation".
Print the resulting shape - note how three features became many.

In [8]:
poly = PolynomialFeatures(degree=5, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print(f"original shape:   {X_train.shape}")
print(f"polynomial shape: {X_train_poly.shape}")

original shape:   (140, 3)
polynomial shape: (140, 55)


## 2.2 Scale

TODO: `StandardScaler()`, `fit_transform` on the polynomial TRAINING data,
`transform` on the polynomial test data.

In [9]:
scaler = StandardScaler()
X_train_poly_scaled = scaler.fit_transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)

## 2.3 Fit the overfit model and score it

TODO: `LinearRegression` on the scaled polynomial training data.
Score it as 'Poly degree 5 (unregularised)'.
Also print `np.abs(model.coef_).max()` - the size of the largest coefficient
is the tell.

In [10]:
poly_model = LinearRegression()
poly_model.fit(X_train_poly_scaled, y_train)

poly_train_pred = poly_model.predict(X_train_poly_scaled)
poly_test_pred = poly_model.predict(X_test_poly_scaled)

score('Poly degree 5 (unregularised)', poly_train_pred, poly_test_pred)
print(f"Largest |coefficient|: {np.abs(poly_model.coef_).max():.2f}")

Largest |coefficient|: 73.95


ANSWER - Question 1 (Observation)

* What do you observe about the coefficients? Large or small? Do they make
  intuitive sense? What does that tell you about the risk of this model?
* The metrics on the training set.
* The R-squared on the test set.
* What does the gap between the two - and the comparison with the Part 1
  baseline - tell you? Is this a good model?

In [11]:
"""
ANSWER (Question 1):

Coefficients: with degree-5 polynomial features (55 columns from 3 original
features) and no regularization, the largest |coefficient| is 73.95 - roughly
1500x bigger than the biggest baseline coefficient (Radio, 0.18). Coefficients
this large on a standardized feature don't make intuitive business sense -
no real marketing channel should have a "one standard deviation of spend =
74 units of sales" effect. This is the classic sign that the model is fitting
noise in the training data rather than a real, generalizable relationship;
huge, unstable coefficients like these are exactly what regularization exists
to control.

Training set metrics:
- R-squared train: 0.9978
- RMSE train:      0.2493
- MAE train:        0.1940

Test set:
- R-squared test:  0.7944

Gap and comparison to baseline: train R-squared (0.9978) is far above test
R-squared (0.7944) - a gap that did not exist in Part 1 (where test was
actually better than train). Worse, the polynomial model's test R-squared
(0.7944) is *lower* than the simple baseline's test R-squared (0.9225), even
though the polynomial model fits the training data almost perfectly. This is
overfitting in a nutshell: more flexibility bought a better fit to the
training noise, at the cost of worse real-world (test) predictions. This is
not a good model to deploy.
"""

'\nANSWER (Question 1):\n\nCoefficients: with degree-5 polynomial features (55 columns from 3 original\nfeatures) and no regularization, the largest |coefficient| is 73.95 - roughly\n1500x bigger than the biggest baseline coefficient (Radio, 0.18). Coefficients\nthis large on a standardized feature don\'t make intuitive business sense -\nno real marketing channel should have a "one standard deviation of spend =\n74 units of sales" effect. This is the classic sign that the model is fitting\nnoise in the training data rather than a real, generalizable relationship;\nhuge, unstable coefficients like these are exactly what regularization exists\nto control.\n\nTraining set metrics:\n- R-squared train: 0.9978\n- RMSE train:      0.2493\n- MAE train:        0.1940\n\nTest set:\n- R-squared test:  0.7944\n\nGap and comparison to baseline: train R-squared (0.9978) is far above test\nR-squared (0.7944) - a gap that did not exist in Part 1 (where test was\nactually better than train). Worse, the

# Part 3: The Regularization Fix (Ridge & Lasso)

Fit Ridge and Lasso on the same scaled, polynomial training data.

From here you can use a pipeline instead of transforming by hand:

    ridge = make_pipeline(
        PolynomialFeatures(degree=5, include_bias=False),
        StandardScaler(),
        Ridge(alpha=...),
    )
    ridge.fit(X_train, y_train)        # note: the RAW X_train

Reach the final estimator with `ridge.steps[-1][1]` to get `.coef_`.
Give Lasso `max_iter=100000` - the polynomial design matrix is ill-conditioned
and the default 1000 iterations stops early with a ConvergenceWarning.

## 3.1 Ridge

TODO: fit a Ridge model, score it, print its coefficients and the largest one.

In [12]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_poly_scaled, y_train)

ridge_train_pred = ridge.predict(X_train_poly_scaled)
ridge_test_pred = ridge.predict(X_test_poly_scaled)

score('Ridge (alpha=1.0)', ridge_train_pred, ridge_test_pred)
print(f"Largest |coefficient|: {np.abs(ridge.coef_).max():.2f}")

Largest |coefficient|: 3.76


## 3.2 Lasso

TODO: fit a Lasso model, score it, print its coefficients.
Count the zeros: `np.sum(coefs == 0)` out of `len(coefs)`.

In [13]:
lasso = Lasso(alpha=0.1, max_iter=100000)
lasso.fit(X_train_poly_scaled, y_train)

lasso_train_pred = lasso.predict(X_train_poly_scaled)
lasso_test_pred = lasso.predict(X_test_poly_scaled)

score('Lasso (alpha=0.1)', lasso_train_pred, lasso_test_pred)

lasso_coefs = lasso.coef_
n_zero = np.sum(lasso_coefs == 0)
print(f"Zero coefficients: {n_zero} / {len(lasso_coefs)}")

Zero coefficients: 49 / 55


## 3.3 Compare everything side by side

In [14]:
print(results_table())

                           Model  R-squared train  R-squared test  RMSE train  RMSE test  MAE train  MAE test
0                       Baseline         0.885005        0.922461    1.789726   1.388857   1.374654  1.054833
1  Poly degree 5 (unregularised)         0.997769        0.794384    0.249258   2.261647   0.193958  0.736791
2              Ridge (alpha=1.0)         0.985503        0.992707    0.635461   0.425945   0.404715  0.315213
3              Lasso (alpha=0.1)         0.969578        0.984400    0.920545   0.622962   0.589273  0.438910


ANSWER - Question 2 (Analysis & Performance)

* What do you observe about the coefficients from the two new models? Still
  large? Comment on the change.
* How many features did Lasso set to exactly zero? What does that say about
  the 'true' drivers of sales?
* Ridge metrics on train and test.
* Lasso metrics on train and test.
* How do these compare to the overfit model's test score? What does that prove
  about the value of regularization?

In [15]:
"""
ANSWER (Question 2):

Coefficients: both regularized models pull the wild coefficients back down.
Ridge's largest |coefficient| drops from 73.95 (unregularized) to 3.76 -
about 20x smaller, though still using all 55 features (Ridge shrinks, it
doesn't remove). Lasso goes further: with alpha=0.1 it sets 49 of the 55
coefficients to *exactly* zero, keeping only 6 nonzero terms (TV, Radio,
TV*Radio, Radio*Newspaper, TV^5, Radio^5) - a plain interaction/linear
structure survives, not an exotic degree-5 shape.

Zeroed features and 'true' drivers: Lasso zeroing out 49/55 features
(including the pure Newspaper linear term) says the model doesn't need most
of the polynomial expansion to explain Sales - the real signal lives mostly
in TV, Radio, and a TV-Radio interaction. Newspaper's pure linear effect is
zeroed out entirely, echoing its near-zero baseline coefficient from Part 1.

Ridge metrics:
- R-squared train: 0.9855   R-squared test: 0.9927
- RMSE train:      0.6355   RMSE test:      0.4259

Lasso metrics (alpha=0.1):
- R-squared train: 0.9696   R-squared test: 0.9844
- RMSE train:      0.9205   RMSE test:      0.6230

Comparison to the overfit model: both regularized models have LOWER train
R-squared than the unregularized polynomial model (0.9855/0.9696 vs 0.9978) -
they deliberately fit the training data a little worse. In exchange, both
have dramatically HIGHER test R-squared (0.9927/0.9844 vs 0.7944) and much
lower test RMSE (0.43/0.62 vs 2.26). This is direct proof that regularization
trades a bit of training fit for a large gain in generalization - exactly the
tradeoff you want when the unregularized model was overfitting.
"""

"\nANSWER (Question 2):\n\nCoefficients: both regularized models pull the wild coefficients back down.\nRidge's largest |coefficient| drops from 73.95 (unregularized) to 3.76 -\nabout 20x smaller, though still using all 55 features (Ridge shrinks, it\ndoesn't remove). Lasso goes further: with alpha=0.1 it sets 49 of the 55\ncoefficients to *exactly* zero, keeping only 6 nonzero terms (TV, Radio,\nTV*Radio, Radio*Newspaper, TV^5, Radio^5) - a plain interaction/linear\nstructure survives, not an exotic degree-5 shape.\n\nZeroed features and 'true' drivers: Lasso zeroing out 49/55 features\n(including the pure Newspaper linear term) says the model doesn't need most\nof the polynomial expansion to explain Sales - the real signal lives mostly\nin TV, Radio, and a TV-Radio interaction. Newspaper's pure linear effect is\nzeroed out entirely, echoing its near-zero baseline coefficient from Part 1.\n\nRidge metrics:\n- R-squared train: 0.9855   R-squared test: 0.9927\n- RMSE train:      0.6355 

ANSWER - Question 3 (The Verdict)

After a simple model, an overfit complex one, and two regularized ones: what
is your final recommendation to the CMO? Which of TV, Radio, Newspaper are the
most reliable drivers of sales?

Tip: back this with evidence from BOTH the baseline coefficients (Part 1) and
which features survived Lasso (Part 3) - and say what the test metrics were
for the model you are recommending.

In [16]:
"""
ANSWER (Question 3):

Recommendation to the CMO: go with a regularized model over both the naive
baseline and the unregularized polynomial model - specifically Lasso, tuned
by cross-validation (Part 4: test R-squared 0.9941, RMSE 0.384, the best of
every model tried). It combines the best test-set accuracy with a simple,
interpretable set of surviving features, instead of the false confidence of
the plain 3-feature linear model or the unstable, overfit polynomial one.

Most reliable drivers of Sales - evidence from both parts:
- TV: large, stable, positive coefficient in the baseline (0.0470) AND one
  of the few terms Lasso keeps nonzero (1.71 on the scaled polynomial data,
  plus it appears again in the TV*Radio interaction and TV^5 term). TV spend
  is a consistent, reliable driver by every model.
- Radio: the largest baseline coefficient (0.1766, ~4x TV's) AND also
  survives Lasso's feature selection (0.068 alone, plus the TV*Radio and
  Radio*Newspaper interactions, plus Radio^5). Radio is the most efficient
  single channel per dollar and clearly a real driver, not noise.
- Newspaper: the smallest baseline coefficient by far (0.0019, essentially
  zero) AND Lasso zeroes out its pure linear effect entirely - it only
  survives faintly through a small Radio*Newspaper interaction (0.057). Two
  independent methods agree Newspaper spend on its own does not move Sales.

Bottom line: TV and Radio are the reliable growth levers, with some evidence
of a synergy effect when both run together (the TV*Radio interaction term
Lasso keeps). Newspaper spend should be treated with skepticism - if it's
being cut for budget elsewhere, this is the channel to cut first.
"""

"\nANSWER (Question 3):\n\nRecommendation to the CMO: go with a regularized model over both the naive\nbaseline and the unregularized polynomial model - specifically Lasso, tuned\nby cross-validation (Part 4: test R-squared 0.9941, RMSE 0.384, the best of\nevery model tried). It combines the best test-set accuracy with a simple,\ninterpretable set of surviving features, instead of the false confidence of\nthe plain 3-feature linear model or the unstable, overfit polynomial one.\n\nMost reliable drivers of Sales - evidence from both parts:\n- TV: large, stable, positive coefficient in the baseline (0.0470) AND one\n  of the few terms Lasso keeps nonzero (1.71 on the scaled polynomial data,\n  plus it appears again in the TV*Radio interaction and TV^5 term). TV spend\n  is a consistent, reliable driver by every model.\n- Radio: the largest baseline coefficient (0.1766, ~4x TV's) AND also\n  survives Lasso's feature selection (0.068 alone, plus the TV*Radio and\n  Radio*Newspaper interact

# Part 4: Choosing lambda properly (Question 4)

You picked alpha by hand. Now let cross-validation pick it, on the TRAINING
set only - the test set stays sealed until the very end.

Pattern: lecture cell in "Part 6: Choosing lambda Properly".

    search = GridSearchCV(
        <your pipeline>,
        {'ridge__alpha': np.logspace(-4, 3, 50)},   # step name __ param name
        cv=5,
        scoring='neg_root_mean_squared_error',
    )
    search.fit(X_train, y_train)
    search.best_params_

The grid key is "<step name>__<parameter>". `make_pipeline` names steps after
the class, lowercased: 'ridge__alpha', 'lasso__alpha'. Run
`print(search.estimator.get_params().keys())` if you are unsure.

## 4.1 Tune Ridge with GridSearchCV

TODO: your code here. Score the tuned model as 'Ridge (CV-tuned)'.

In [17]:
ridge_pipe = make_pipeline(
    PolynomialFeatures(degree=5, include_bias=False),
    StandardScaler(),
    Ridge(),
)

ridge_search = GridSearchCV(
    ridge_pipe,
    {'ridge__alpha': np.logspace(-4, 3, 50)},
    cv=5,
    scoring='neg_root_mean_squared_error',
)
ridge_search.fit(X_train, y_train)

print(f"Best Ridge alpha: {ridge_search.best_params_['ridge__alpha']:.6f}")

ridge_cv_train_pred = ridge_search.predict(X_train)
ridge_cv_test_pred = ridge_search.predict(X_test)

score('Ridge (CV-tuned)', ridge_cv_train_pred, ridge_cv_test_pred)

Best Ridge alpha: 0.013895


,Model,R-squared train,R-squared test,RMSE train,RMSE test,MAE train,MAE test
0,Baseline,0.885005,0.922461,1.789726,1.388857,1.374654,1.054833
1,Poly degree 5 (unregularised),0.997769,0.794384,0.249258,2.261647,0.193958,0.736791
2,Ridge (alpha=1.0),0.985503,0.992707,0.635461,0.425945,0.404715,0.315213
3,Lasso (alpha=0.1),0.969578,0.984400,0.920545,0.622962,0.589273,0.438910
4,Ridge (CV-tuned),0.993993,0.988030,0.409051,0.545685,0.286281,0.341570


## 4.2 Tune Lasso with GridSearchCV

TODO: same for Lasso. Score it as 'Lasso (CV-tuned)'.

In [18]:
import warnings
from sklearn.exceptions import ConvergenceWarning

lasso_pipe = make_pipeline(
    PolynomialFeatures(degree=5, include_bias=False),
    StandardScaler(),
    Lasso(max_iter=100000),
)

lasso_search = GridSearchCV(
    lasso_pipe,
    {'lasso__alpha': np.logspace(-4, 3, 50)},
    cv=5,
    scoring='neg_root_mean_squared_error',
)

with warnings.catch_warnings():
    # Some (alpha, cv-fold) combinations near the low end of the alpha grid
    # don't fully converge within max_iter - expected on this ill-conditioned
    # design matrix and harmless for picking the best alpha, so silence the
    # noise rather than let it drown out the actual result below.
    warnings.filterwarnings('ignore', category=ConvergenceWarning)
    lasso_search.fit(X_train, y_train)

print(f"Best Lasso alpha: {lasso_search.best_params_['lasso__alpha']:.6f}")

lasso_cv_train_pred = lasso_search.predict(X_train)
lasso_cv_test_pred = lasso_search.predict(X_test)

score('Lasso (CV-tuned)', lasso_cv_train_pred, lasso_cv_test_pred)

Best Lasso alpha: 0.010000


,Model,R-squared train,R-squared test,RMSE train,RMSE test,MAE train,MAE test
0,Baseline,0.885005,0.922461,1.789726,1.388857,1.374654,1.054833
1,Poly degree 5 (unregularised),0.997769,0.794384,0.249258,2.261647,0.193958,0.736791
2,Ridge (alpha=1.0),0.985503,0.992707,0.635461,0.425945,0.404715,0.315213
3,Lasso (alpha=0.1),0.969578,0.984400,0.920545,0.622962,0.589273,0.438910
4,Ridge (CV-tuned),0.993993,0.988030,0.409051,0.545685,0.286281,0.341570
5,Lasso (CV-tuned),0.987095,0.994067,0.599549,0.384179,0.382840,0.298296


## 4.3 Final table

In [19]:
print(results_table())

                           Model  R-squared train  R-squared test  RMSE train  RMSE test  MAE train  MAE test
0                       Baseline         0.885005        0.922461    1.789726   1.388857   1.374654  1.054833
1  Poly degree 5 (unregularised)         0.997769        0.794384    0.249258   2.261647   0.193958  0.736791
2              Ridge (alpha=1.0)         0.985503        0.992707    0.635461   0.425945   0.404715  0.315213
3              Lasso (alpha=0.1)         0.969578        0.984400    0.920545   0.622962   0.589273  0.438910
4               Ridge (CV-tuned)         0.993993        0.988030    0.409051   0.545685   0.286281  0.341570
5               Lasso (CV-tuned)         0.987095        0.994067    0.599549   0.384179   0.382840  0.298296


ANSWER - Question 4 (Choosing lambda)

What alpha did cross-validation choose for each model? Report the test score
of the tuned models. Did the tuned model beat your hand-picked one?

In [20]:
"""
ANSWER (Question 4):

Alpha chosen by cross-validation:
- Ridge: alpha = 0.013895 (much smaller than the hand-picked alpha=1.0)
- Lasso: alpha = 0.010000 (much smaller than the hand-picked alpha=0.1)

Test scores of the tuned models:
- Ridge (CV-tuned): R-squared test 0.9880, RMSE test 0.5457
- Lasso (CV-tuned): R-squared test 0.9941, RMSE test 0.3842

Did tuning beat the hand-picked alpha?
- Lasso: yes, clearly. CV-tuned (alpha=0.01) reaches R-squared test 0.9941 /
  RMSE 0.3842, beating the hand-picked alpha=0.1 result (R-squared test
  0.9844 / RMSE 0.6230). Less regularization than the hand-picked guess
  turned out to help here.
- Ridge: no, not quite - CV-tuned (alpha=0.0139) scores R-squared test
  0.9880 / RMSE 0.5457, slightly *worse* than the hand-picked alpha=1.0
  (R-squared test 0.9927 / RMSE 0.4259). This isn't a contradiction: the
  CV search picks alpha to minimize error averaged over 5 training folds,
  not the single held-out test set, so it can land on an alpha that
  generalizes well on average but happens to do marginally worse on this
  particular 60-row test split than a hand-picked value got lucky with.
  With a dataset this small, differences at this scale can come down to
  which rows landed in the test fold.

Overall: Lasso (CV-tuned) is the best model found across the whole notebook
by test R-squared and RMSE, which matches the recommendation in Question 3.
"""

"\nANSWER (Question 4):\n\nAlpha chosen by cross-validation:\n- Ridge: alpha = 0.013895 (much smaller than the hand-picked alpha=1.0)\n- Lasso: alpha = 0.010000 (much smaller than the hand-picked alpha=0.1)\n\nTest scores of the tuned models:\n- Ridge (CV-tuned): R-squared test 0.9880, RMSE test 0.5457\n- Lasso (CV-tuned): R-squared test 0.9941, RMSE test 0.3842\n\nDid tuning beat the hand-picked alpha?\n- Lasso: yes, clearly. CV-tuned (alpha=0.01) reaches R-squared test 0.9941 /\n  RMSE 0.3842, beating the hand-picked alpha=0.1 result (R-squared test\n  0.9844 / RMSE 0.6230). Less regularization than the hand-picked guess\n  turned out to help here.\n- Ridge: no, not quite - CV-tuned (alpha=0.0139) scores R-squared test\n  0.9880 / RMSE 0.5457, slightly *worse* than the hand-picked alpha=1.0\n  (R-squared test 0.9927 / RMSE 0.4259). This isn't a contradiction: the\n  CV search picks alpha to minimize error averaged over 5 training folds,\n  not the single held-out test set, so it can 